In [2]:
import sys
import os
from datetime import date

sys.path.append(os.path.abspath("../src"))

from domain import AppCredential
from domain.config import DB_URL, ENVIRONMENT
from domain.unit_of_work import UnitOfWorkFactory
from domain.models import Category, Employee, Menu, Dish, Sale, Allergen

uow_factory = UnitOfWorkFactory(DB_URL)
print(f"UnitOfWorkFactory inicialitzat correctament. Entorn: {ENVIRONMENT}")
print(f"URL: {DB_URL}")

UnitOfWorkFactory inicialitzat correctament. Entorn: test
URL: sqlite:///:memory:


In [3]:
# Creem les taules des de zero per a les proves en memòria
if ENVIRONMENT == "test":
    from domain.db import Base
    Base.metadata.drop_all(uow_factory.engine)
    Base.metadata.create_all(uow_factory.engine)
    print("Taules creades en memòria correctament.")

Taules creades en memòria correctament.


## 1. CategoryRepository

In [4]:
# add() - Creació de categories
print("=== CategoryRepository: add() ===")
with uow_factory.create() as uow:
    cat1 = Category(category_name="Arrossos")
    cat2 = Category(category_name="Carns")
    cat3 = Category(category_name="Infusions")
    uow.categories.add(cat1)
    uow.categories.add(cat2)
    uow.categories.add(cat3)

print(f"Categoria creada - ID: {cat1.id}, Nom: {cat1.category_name}")
print(f"Categoria creada - ID: {cat2.id}, Nom: {cat2.category_name}")
print(f"Categoria creada - ID: {cat3.id}, Nom: {cat3.category_name}")

=== CategoryRepository: add() ===
Categoria creada - ID: 1, Nom: Arrossos
Categoria creada - ID: 2, Nom: Carns
Categoria creada - ID: 3, Nom: Infusions


In [5]:
# Prova de validació - Categoria duplicada
print("=== CategoryRepository: prova de validació ===")
with uow_factory.create() as uow:
    try:
        duplicate_cat = Category(category_name="Arrossos")
        uow.categories.add(duplicate_cat)
        uow.commit()
        print("ERROR: S'ha permès crear una categoria duplicada!")
    except ValueError as e:
        print(f"Validació correcta! Missatge: {e}")

=== CategoryRepository: prova de validació ===
Validació correcta! Missatge: Category 'Arrossos' already exists.


In [6]:
# get() - Llegir per ID
print("=== CategoryRepository: get() ===")
with uow_factory.create() as uow:
    categoria = uow.categories.get(cat1.id)

print(f"Categoria trobada: {categoria}")

if categoria is not None:
    print(f"  Nom: {categoria.category_name}")
else:
    print("  No s'ha trobat la categoria.")

=== CategoryRepository: get() ===
Categoria trobada: Arrossos
  Nom: Arrossos


In [7]:
# get_all() - Llegir totes les categories
print("=== CategoryRepository: get_all() ===")
with uow_factory.create() as uow:
    totes_categories = uow.categories.get_all()

print(f"Total categories a la BD: {len(totes_categories)}")
for cat in totes_categories:
    print(f"  - {cat}")

=== CategoryRepository: get_all() ===
Total categories a la BD: 3
  - Arrossos
  - Carns
  - Infusions


In [8]:
# update() - Modificar una categoria
print("=== CategoryRepository: update() ===")
with uow_factory.create() as uow:
    cat_a_modificar = uow.categories.get(cat1.id)
    cat_a_modificar.category_name = "Arrossos i Pasta"
    uow.categories.update(cat_a_modificar)

with uow_factory.create() as uow:
    cat_actualitzada = uow.categories.get(cat1.id)
print(f"Categoria actualitzada: {cat_actualitzada.category_name}")

=== CategoryRepository: update() ===
Categoria actualitzada: Arrossos i Pasta


In [9]:
# update() - Validació de nom duplicat
print("=== CategoryRepository: update() validació ===")
with uow_factory.create() as uow:
    try:
        cat_a_modificar = uow.categories.get(cat1.id)
        cat_a_modificar.category_name = "Carns"  # nom que ja existeix a cat2
        uow.categories.update(cat_a_modificar)
        print("ERROR: S'ha permès un nom duplicat!")
    except ValueError as e:
        print(f"Validació correcta! Missatge: {e}")

=== CategoryRepository: update() validació ===
Validació correcta! Missatge: Category name 'Carns' is already taken.


In [10]:
# get_by_name() - Cerca per nom
print("=== CategoryRepository: get_by_name() ===")
with uow_factory.create() as uow:
    resultat = uow.categories.get_by_name("Carns")

print(f"Categoria trobada per nom: {resultat}")
if resultat is None:
    print("  No s'ha trobat cap categoria amb aquest nom.")

=== CategoryRepository: get_by_name() ===
Categoria trobada per nom: Carns


In [11]:
# delete() - Eliminar una categoria
print("=== CategoryRepository: delete() ===")
with uow_factory.create() as uow:
    eliminada = uow.categories.delete(cat3.id)

print(f"Eliminació realitzada: {eliminada}")

with uow_factory.create() as uow:
    comprovacio = uow.categories.get(cat3.id)
print(f"Categoria eliminada (hauria de ser None): {comprovacio}")

=== CategoryRepository: delete() ===
Eliminació realitzada: None
Categoria eliminada (hauria de ser None): None


## 2. AllergenRepository

In [12]:
# add() - Creació d'al·lèrgens
print("=== AllergenRepository: add() ===")
with uow_factory.create() as uow:
    all1 = Allergen(allergen_name="Mostassa")
    all2 = Allergen(allergen_name="Api")
    all3 = Allergen(allergen_name="Llúpol")
    uow.allergens.add(all1)
    uow.allergens.add(all2)
    uow.allergens.add(all3)

print(f"Al·lèrgens creats - IDs: {all1.id}, {all2.id}, {all3.id}")

=== AllergenRepository: add() ===
Al·lèrgens creats - IDs: 1, 2, 3


In [13]:
# get(), get_all(), get_by_name()
print("=== AllergenRepository: get() / get_all() / get_by_name() ===")
with uow_factory.create() as uow:
    allergen = uow.allergens.get(all1.id)
    tots_allergens = uow.allergens.get_all()
    per_nom = uow.allergens.get_by_name("Api")

print(f"get(): {allergen}")
print(f"get_all() - total: {len(tots_allergens)}")
print(f"get_by_name('Api'): {per_nom}")

=== AllergenRepository: get() / get_all() / get_by_name() ===
get(): Mostassa
get_all() - total: 3
get_by_name('Api'): Api


In [14]:
# update() i delete()
print("=== AllergenRepository: update() i delete() ===")
with uow_factory.create() as uow:
    a = uow.allergens.get(all3.id)
    a.allergen_name = "Diòxid de sofre"
    uow.allergens.update(a)

with uow_factory.create() as uow:
    allergen_actualitzat = uow.allergens.get(all3.id)
print(f"Al·lèrgen actualitzat: {allergen_actualitzat.allergen_name}")

=== AllergenRepository: update() i delete() ===
Al·lèrgen actualitzat: Diòxid de sofre


## 3. DishRepository

In [15]:
# add() - Creació de plats
# Fem servir cat1 i cat2 que hem creat en aquest mateix notebook
print("=== DishRepository: add() ===")
with uow_factory.create() as uow:
    dish1 = Dish(dish_name="Arròs negre", description="Arròs amb tinta de calamar", is_active=True, id_category=cat1.id)
    dish2 = Dish(dish_name="Entrecot de vedella", description="Vedella de 250g amb guarnició", is_active=True, id_category=cat2.id)
    dish3 = Dish(dish_name="Arròs a banda", description="Arròs de mar tradicional", is_active=False, id_category=cat1.id)
    uow.dishes.add(dish1)
    uow.dishes.add(dish2)
    uow.dishes.add(dish3)

print(f"Plats creats - IDs: {dish1.id}, {dish2.id}, {dish3.id}")

=== DishRepository: add() ===
Plats creats - IDs: 1, 2, 3


In [16]:
# get() i get_all()
print("=== DishRepository: get() / get_all() ===")
with uow_factory.create() as uow:
    plat = uow.dishes.get(dish1.id)
    tots_plats = uow.dishes.get_all()

print(f"get(): {plat}")
print(f"get_all() - total: {len(tots_plats)}")

=== DishRepository: get() / get_all() ===
get(): Arròs negre
get_all() - total: 3


In [17]:
# get_active_dishes() - Cerca plats actius
print("=== DishRepository: get_active_dishes() ===")
with uow_factory.create() as uow:
    actius = uow.dishes.get_active_dishes()

print(f"Plats actius trobats: {len(actius)}")
for d in actius:
    print(f"  - {d.dish_name} (actiu: {d.is_active})")
print("Nota: 'Arròs a banda' NO hauria d'aparèixer perquè is_active=False")

=== DishRepository: get_active_dishes() ===
Plats actius trobats: 2
  - Arròs negre (actiu: True)
  - Entrecot de vedella (actiu: True)
Nota: 'Arròs a banda' NO hauria d'aparèixer perquè is_active=False


In [18]:
# get_by_category() - Cerca per categoria
print("=== DishRepository: get_by_category() ===")
with uow_factory.create() as uow:
    plats_cat1 = uow.dishes.get_by_category(cat1.id)

print(f"Plats de la categoria '{cat1.category_name}' (ID {cat1.id}):")
for d in plats_cat1:
    print(f"  - {d.dish_name}")

=== DishRepository: get_by_category() ===
Plats de la categoria 'Arrossos' (ID 1):
  - Arròs negre
  - Arròs a banda


In [19]:
# get_by_name() - Cerca per nom
print("=== DishRepository: get_by_name() ===")
with uow_factory.create() as uow:
    resultats = uow.dishes.get_by_name("Arròs")

print(f"Plats trobats amb 'Arròs': {resultats}")

=== DishRepository: get_by_name() ===
Plats trobats amb 'Arròs': [Dish(id=1, name='Arròs negre'), Dish(id=3, name='Arròs a banda')]


In [20]:
# add_allergen_to_dish() - Afegir al·lèrgen a un plat
print("=== DishRepository: add_allergen_to_dish() ===")
with uow_factory.create() as uow:
    allergen_mostassa = uow.allergens.get(all1.id)
    resultat = uow.dishes.add_allergen_to_dish(dish1.id, allergen_mostassa)

print(f"Al·lèrgen afegit al plat: {resultat}")

with uow_factory.create() as uow:
    d = uow.dishes.get(dish1.id)
    print(f"Al·lèrgens de '{d.dish_name}': {d.allergens}")

=== DishRepository: add_allergen_to_dish() ===
Al·lèrgen afegit al plat: True
Al·lèrgens de 'Arròs negre': [Allergen(id=1, name='Mostassa')]


In [21]:
# update() - Activar un plat que estava inactiu
print("=== DishRepository: update() ===")
with uow_factory.create() as uow:
    d = uow.dishes.get(dish3.id)
    d.is_active = True
    uow.dishes.update(d)

with uow_factory.create() as uow:
    plat_actualitzat = uow.dishes.get(dish3.id)
print(f"Plat '{plat_actualitzat.dish_name}' - is_active actualitzat a: {plat_actualitzat.is_active}")

=== DishRepository: update() ===
Plat 'Arròs a banda' - is_active actualitzat a: True


## 4. EmployeeRepository

In [22]:
# add() - Creació d'empleats
# DNIs completament diferents als del populate (que usa 11111111A..66666666F)
print("=== EmployeeRepository: add() ===")
with uow_factory.create() as uow:
    emp1 = Employee(
        name="Carles", surname="Bosch", national_id="77777777G", role="Cambrer",
        credentials=AppCredential(email="carles@test.cat", password_hash="hash_carles")
    )
    emp2 = Employee(
        name="Núria", surname="Camps", national_id="88888888H", role="Cuinera",
        credentials=AppCredential(email="nuria@test.cat", password_hash="hash_nuria")
    )
    emp3 = Employee(
        name="Ferran", surname="Domènech", national_id="99999999I", role="Cambrer"
    )
    uow.employees.add(emp1)
    uow.employees.add(emp2)
    uow.employees.add(emp3)

print(f"Empleats creats - IDs: {emp1.id}, {emp2.id}, {emp3.id}")

=== EmployeeRepository: add() ===
Empleats creats - IDs: 1, 2, 3


In [23]:
# Prova de validació - DNI duplicat
print("=== EmployeeRepository: prova de validació ===")
with uow_factory.create() as uow:
    try:
        emp_duplicat = Employee(
            name="Impostor", surname="Gimenez", national_id="77777777G", role="Cambrer"
        )
        uow.employees.add(emp_duplicat)
        uow.commit()
        print("ERROR: S'ha permès crear un empleat amb DNI duplicat!")
    except ValueError as e:
        print(f"Validació correcta! Missatge: {e}")

=== EmployeeRepository: prova de validació ===
Validació correcta! Missatge: Employee with national ID '77777777G' already exists.


In [24]:
# get() - Llegir per ID
print("=== EmployeeRepository: get() ===")
with uow_factory.create() as uow:
    empleat = uow.employees.get(emp1.id)

print(f"Empleat trobat: {empleat}")
if empleat is not None:
    print(f"  Nom: {empleat.name}, Rol: {empleat.role}")

=== EmployeeRepository: get() ===
Empleat trobat: 1: Bosch, Carles (Cambrer)
  Nom: Carles, Rol: Cambrer


In [25]:
# get_all() - Llegir tots els empleats
print("=== EmployeeRepository: get_all() ===")
with uow_factory.create() as uow:
    tots_empleats = uow.employees.get_all()

print(f"Total empleats a la BD: {len(tots_empleats)}")
for emp in tots_empleats:
    print(f"  - {emp}")

=== EmployeeRepository: get_all() ===
Total empleats a la BD: 3
  - 1: Bosch, Carles (Cambrer)
  - 2: Camps, Núria (Cuinera)
  - 3: Domènech, Ferran (Cambrer)


In [26]:
# update() - Modificar un empleat
print("=== EmployeeRepository: update() ===")
with uow_factory.create() as uow:
    emp_a_modificar = uow.employees.get(emp1.id)
    emp_a_modificar.role = "Encarregat"
    uow.employees.update(emp_a_modificar)

with uow_factory.create() as uow:
    emp_actualitzat = uow.employees.get(emp1.id)
print(f"Empleat '{emp_actualitzat.name}' - rol actualitzat a: {emp_actualitzat.role}")

=== EmployeeRepository: update() ===
Empleat 'Carles' - rol actualitzat a: Encarregat


In [27]:
# update() - Validació de DNI duplicat
print("=== EmployeeRepository: update() validació ===")
with uow_factory.create() as uow:
    try:
        dni_nou = "88888888H"  # DNI que ja existeix a emp2
        if uow.employees.national_id_exists(dni_nou):
            raise ValueError(f"No es pot actualitzar. Ja existeix un altre empleat amb el DNI: {dni_nou}")
        emp_a_modificar = uow.employees.get(emp1.id)
        emp_a_modificar.national_id = dni_nou
        uow.employees.update(emp_a_modificar)
        print("ERROR: S'ha permès un DNI duplicat!")
    except ValueError as e:
        print(f"Validació correcta! Missatge: {e}")

=== EmployeeRepository: update() validació ===
Validació correcta! Missatge: No es pot actualitzar. Ja existeix un altre empleat amb el DNI: 88888888H


In [28]:
# get_by_name() - Cerca per nom o cognom
print("=== EmployeeRepository: get_by_name() ===")
with uow_factory.create() as uow:
    resultats = uow.employees.get_by_name("Núria", "")

print(f"Empleats trobats amb nom 'Núria': {resultats}")

=== EmployeeRepository: get_by_name() ===
Empleats trobats amb nom 'Núria': [Employee(id=1, name='Carles', role='Encarregat'), Employee(id=2, name='Núria', role='Cuinera'), Employee(id=3, name='Ferran', role='Cambrer')]


In [29]:
# get_by_role() - Cerca per rol
print("=== EmployeeRepository: get_by_role() ===")
with uow_factory.create() as uow:
    cambrers = uow.employees.get_by_role("Cambrer")

    print(f"Cambrers trobats: {cambrers}")
# Nota: si existeixen dades del populate, també apareixeran els seus cambrers

=== EmployeeRepository: get_by_role() ===
Cambrers trobats: [Employee(id=3, name='Ferran', role='Cambrer')]


In [30]:
# national_id_exists() - Validació de DNI
print("=== EmployeeRepository: national_id_exists() ===")
with uow_factory.create() as uow:
    existeix_si = uow.employees.national_id_exists("77777777G")
    existeix_no = uow.employees.national_id_exists("00000000Z")

    print(f"DNI '77777777G' existeix: {existeix_si} (hauria de ser True)")
    print(f"DNI '00000000Z' existeix: {existeix_no} (hauria de ser False)")

=== EmployeeRepository: national_id_exists() ===
DNI '77777777G' existeix: True (hauria de ser True)
DNI '00000000Z' existeix: False (hauria de ser False)


In [31]:
# get_paginated() - Paginació
print("=== EmployeeRepository: get_paginated() ===")
with uow_factory.create() as uow:
    pagina1 = uow.employees.get_paginated(page=1, page_size=2)
    pagina2 = uow.employees.get_paginated(page=2, page_size=2)

    print(f"Pàgina 1 (2 per pàgina): {pagina1}")
    print(f"Pàgina 2 (2 per pàgina): {pagina2}")

=== EmployeeRepository: get_paginated() ===
Pàgina 1 (2 per pàgina): [Employee(id=1, name='Carles', role='Encarregat'), Employee(id=2, name='Núria', role='Cuinera')]
Pàgina 2 (2 per pàgina): [Employee(id=3, name='Ferran', role='Cambrer')]


In [32]:
# delete() - Eliminar un empleat
print("=== EmployeeRepository: delete() ===")
with uow_factory.create() as uow:
    eliminat = uow.employees.delete(emp3.id)

print(f"Eliminació realitzada: {eliminat}")

with uow_factory.create() as uow:
    comprovacio = uow.employees.get(emp3.id)
print(f"Empleat eliminat (hauria de ser None): {comprovacio}")

=== EmployeeRepository: delete() ===
Eliminació realitzada: None
Empleat eliminat (hauria de ser None): None


## 5. MenuRepository

In [33]:
# add() - Creació de menús
print("=== MenuRepository: add() ===")
with uow_factory.create() as uow:
    menu1 = Menu(menu_name="Menú Executiu", price=14.00)
    menu2 = Menu(menu_name="Menú Degustació", price=38.00)
    menu3 = Menu(menu_name="Menú Infantil", price=7.50)
    uow.menus.add(menu1)
    uow.menus.add(menu2)
    uow.menus.add(menu3)

print(f"Menús creats - IDs: {menu1.id}, {menu2.id}, {menu3.id}")

=== MenuRepository: add() ===
Menús creats - IDs: 1, 2, 3


In [34]:
# get() i get_all()
print("=== MenuRepository: get() / get_all() ===")
with uow_factory.create() as uow:
    menu = uow.menus.get(menu1.id)
    tots_menus = uow.menus.get_all()

print(f"get(): {menu}")
print(f"get_all() - total: {len(tots_menus)}")

=== MenuRepository: get() / get_all() ===
get(): Menú Executiu (14.00€)
get_all() - total: 3


In [35]:
# add_dish_to_menu() - Afegir plats al menú
print("=== MenuRepository: add_dish_to_menu() ===")
with uow_factory.create() as uow:
    d1 = uow.dishes.get(dish1.id)
    d2 = uow.dishes.get(dish2.id)
    uow.menus.add_dish_to_menu(menu1.id, d1)
    uow.menus.add_dish_to_menu(menu1.id, d2)

with uow_factory.create() as uow:
    m = uow.menus.get(menu1.id)
    print(f"Plats del '{m.menu_name}': {m.dishes}")

=== MenuRepository: add_dish_to_menu() ===
Plats del 'Menú Executiu': [Dish(id=1, name='Arròs negre'), Dish(id=2, name='Entrecot de vedella')]


In [36]:
# remove_dish_from_menu() - Treure un plat del menú
print("=== MenuRepository: remove_dish_from_menu() ===")
with uow_factory.create() as uow:
    d2 = uow.dishes.get(dish2.id)
    resultat = uow.menus.remove_dish_from_menu(menu1.id, d2)

print(f"Eliminació del plat del menú: {resultat}")
with uow_factory.create() as uow:
    m = uow.menus.get(menu1.id)
    print(f"Plats restants al menú: {m.dishes}")

=== MenuRepository: remove_dish_from_menu() ===
Eliminació del plat del menú: True
Plats restants al menú: [Dish(id=1, name='Arròs negre')]


In [37]:
# get_by_name() - Cerca per nom
print("=== MenuRepository: get_by_name() ===")
with uow_factory.create() as uow:
    resultats = uow.menus.get_by_name("Degustació")

    print(f"Menús trobats amb 'Degustació': {resultats}")

=== MenuRepository: get_by_name() ===
Menús trobats amb 'Degustació': [Menu(id=2, name='Menú Degustació', price=38.00)]


In [38]:
# get_by_price_range() - Cerca per rang de preus
print("=== MenuRepository: get_by_price_range() ===")
with uow_factory.create() as uow:
    assequibles = uow.menus.get_by_price_range(0, 15.00)

print(f"Menús fins a 15€:")
for m in assequibles:
    print(f"  - {m.menu_name}: {m.price}€")
print("Nota: 'Menú Executiu' (14€) i 'Menú Infantil' (7.50€) haurien d'aparèixer")

=== MenuRepository: get_by_price_range() ===
Menús fins a 15€:
  - Menú Executiu: 14.00€
  - Menú Infantil: 7.50€
Nota: 'Menú Executiu' (14€) i 'Menú Infantil' (7.50€) haurien d'aparèixer


In [39]:
# update() - Modificar el preu d'un menú
print("=== MenuRepository: update() ===")
with uow_factory.create() as uow:
    m = uow.menus.get(menu1.id)
    m.price = 15.00
    uow.menus.update(m)

with uow_factory.create() as uow:
    menu_actualitzat = uow.menus.get(menu1.id)
print(f"Preu actualitzat de '{menu_actualitzat.menu_name}': {menu_actualitzat.price}€")

=== MenuRepository: update() ===
Preu actualitzat de 'Menú Executiu': 15.00€


## 6. SaleRepository

In [40]:
# add() - Creació de vendes
# Fem servir emp1 i emp2 (creats en aquest notebook) i dates de maig 2026
# (el populate usa dates de març 2026, no hi ha conflicte)
print("=== SaleRepository: add() ===")
with uow_factory.create() as uow:
    sale1 = Sale(id_menu=menu1.id, id_employee=emp1.id, sale_date=date(2026, 5, 10), quantity_sold=3)
    sale2 = Sale(id_menu=menu2.id, id_employee=emp1.id, sale_date=date(2026, 5, 10), quantity_sold=1)
    sale3 = Sale(id_menu=menu1.id, id_employee=emp2.id, sale_date=date(2026, 5, 11), quantity_sold=5)
    sale4 = Sale(id_menu=menu3.id, id_employee=emp2.id, sale_date=date(2026, 5, 12), quantity_sold=2)
    uow.sales.add(sale1)
    uow.sales.add(sale2)
    uow.sales.add(sale3)
    uow.sales.add(sale4)

print("Vendes creades correctament.")

=== SaleRepository: add() ===
Vendes creades correctament.


In [41]:
# get() - La Sale té clau primària composta (tuple)
print("=== SaleRepository: get() ===")
with uow_factory.create() as uow:
    # La clau primària de Sale és (id_menu, id_employee, sale_date)
    venda = uow.sales.get((menu1.id, emp1.id, date(2026, 5, 10)))

print(f"Venda trobada: {venda}")
if venda is not None:
    print(f"  Quantitat venuda: {venda.quantity_sold}")

=== SaleRepository: get() ===
Venda trobada: Venda del 2026-05-10: Menú ID 1 venut per Empleat ID 1
  Quantitat venuda: 3


In [42]:
# get_all()
print("=== SaleRepository: get_all() ===")
with uow_factory.create() as uow:
    totes_vendes = uow.sales.get_all()

print(f"Total vendes a la BD: {len(totes_vendes)}")
for s in totes_vendes:
    print(f"  - {s}")

=== SaleRepository: get_all() ===
Total vendes a la BD: 4
  - Venda del 2026-05-10: Menú ID 1 venut per Empleat ID 1
  - Venda del 2026-05-10: Menú ID 2 venut per Empleat ID 1
  - Venda del 2026-05-11: Menú ID 1 venut per Empleat ID 2
  - Venda del 2026-05-12: Menú ID 3 venut per Empleat ID 2


In [43]:
# update() - Modificar la quantitat d'una venda
print("=== SaleRepository: update() ===")
with uow_factory.create() as uow:
    s = uow.sales.get((menu1.id, emp1.id, date(2026, 5, 10)))
    s.quantity_sold = 10
    uow.sales.update(s)

with uow_factory.create() as uow:
    venda_actualitzada = uow.sales.get((menu1.id, emp1.id, date(2026, 5, 10)))
print(f"Quantitat actualitzada: {venda_actualitzada.quantity_sold} (abans era 3)")

=== SaleRepository: update() ===
Quantitat actualitzada: 10 (abans era 3)


In [44]:
# get_sales_by_date() - Cerca per data
print("=== SaleRepository: get_sales_by_date() ===")
with uow_factory.create() as uow:
    vendes_dia = uow.sales.get_sales_by_date(date(2026, 5, 10))

print(f"Vendes del 2026-05-10: {vendes_dia}")
print(f"  Total: {len(vendes_dia)} (haurien de ser 2: sale1 i sale2)")

=== SaleRepository: get_sales_by_date() ===
Vendes del 2026-05-10: [Sale(menu=1, emp=1, date=2026-05-10), Sale(menu=2, emp=1, date=2026-05-10)]
  Total: 2 (haurien de ser 2: sale1 i sale2)


In [45]:
# get_sales_by_employee() - Cerca per empleat
print("=== SaleRepository: get_sales_by_employee() ===")
with uow_factory.create() as uow:
    vendes_emp1 = uow.sales.get_sales_by_employee(emp1.id)

print(f"Vendes de l'empleat {emp1.id} ({emp1.name}): {vendes_emp1}")
print(f"  Total: {len(vendes_emp1)} (haurien de ser 2: sale1 i sale2)")

=== SaleRepository: get_sales_by_employee() ===
Vendes de l'empleat 1 (Carles): [Sale(menu=1, emp=1, date=2026-05-10), Sale(menu=2, emp=1, date=2026-05-10)]
  Total: 2 (haurien de ser 2: sale1 i sale2)


In [46]:
# get_sales_by_menu() - Cerca per menú
print("=== SaleRepository: get_sales_by_menu() ===")
with uow_factory.create() as uow:
    vendes_menu1 = uow.sales.get_sales_by_menu(menu1.id)

print(f"Vendes del menú {menu1.id} ('{menu1.menu_name}'): {vendes_menu1}")
print(f"  Total: {len(vendes_menu1)} (haurien de ser 2: sale1 i sale3)")

=== SaleRepository: get_sales_by_menu() ===
Vendes del menú 1 ('Menú Executiu'): [Sale(menu=1, emp=1, date=2026-05-10), Sale(menu=1, emp=2, date=2026-05-11)]
  Total: 2 (haurien de ser 2: sale1 i sale3)


In [47]:
# get_sales_by_date_range() - Cerca per rang de dates
print("=== SaleRepository: get_sales_by_date_range() ===")
with uow_factory.create() as uow:
    vendes_rang = uow.sales.get_sales_by_date_range(date(2026, 5, 10), date(2026, 5, 11))
    vendes_buit = uow.sales.get_sales_by_date_range(date(2026, 6, 1), date(2026, 6, 30))

print(f"Vendes del 10 al 11 de maig: {vendes_rang}")
print(f"  Total: {len(vendes_rang)} (haurien de ser 3: sale1, sale2 i sale3)")
print(f"Vendes del juny (hauria de ser buit): {vendes_buit}")

=== SaleRepository: get_sales_by_date_range() ===
Vendes del 10 al 11 de maig: [Sale(menu=1, emp=1, date=2026-05-10), Sale(menu=2, emp=1, date=2026-05-10), Sale(menu=1, emp=2, date=2026-05-11)]
  Total: 3 (haurien de ser 3: sale1, sale2 i sale3)
Vendes del juny (hauria de ser buit): []


In [48]:
# get_paginated() - Paginació de vendes
print("=== SaleRepository: get_paginated() ===")
with uow_factory.create() as uow:
    pagina1 = uow.sales.get_paginated(page=1, page_size=2)
    pagina2 = uow.sales.get_paginated(page=2, page_size=2)

print(f"Pàgina 1 (2 per pàgina): {pagina1}")
print(f"Pàgina 2 (2 per pàgina): {pagina2}")

=== SaleRepository: get_paginated() ===
Pàgina 1 (2 per pàgina): [Sale(menu=1, emp=1, date=2026-05-10), Sale(menu=2, emp=1, date=2026-05-10)]
Pàgina 2 (2 per pàgina): [Sale(menu=1, emp=2, date=2026-05-11), Sale(menu=3, emp=2, date=2026-05-12)]


In [49]:
# delete() - Eliminar una venda
print("=== SaleRepository: delete() ===")
with uow_factory.create() as uow:
    eliminada = uow.sales.delete((menu3.id, emp2.id, date(2026, 5, 12)))

print(f"Eliminació realitzada: {eliminada}")

with uow_factory.create() as uow:
    comprovacio = uow.sales.get((menu3.id, emp2.id, date(2026, 5, 12)))
print(f"Venda eliminada (hauria de ser None): {comprovacio}")

=== SaleRepository: delete() ===
Eliminació realitzada: None
Venda eliminada (hauria de ser None): None
